# 04 — Embedding data mới: Kaggle acc 0

**Add Input dataset NEW_DATASET_READY → Internet ON → GPU T4 x2 → Save Version / Save & Run All.**
Acc 0 và acc 1 dùng CÙNG dataset; chủ dataset cấp quyền đọc cho acc còn lại.
Đã đặt KAGGLE_ACCOUNT_ID=0. Nếu gói xuất chỉ một acc, chỉ dùng bản acc 0.
Model BGE-M3 và Qwen3-Embedding-8B, runtime/producer từ job; không mount Drive.

Chỉ chạy text thiếu trong lượt đã giao. Mỗi task tối đa 512 text có checkpoint
trong /kaggle/working. Deadline 10 giờ từ cell đầu để kịp lưu Outputs; quota
tài khoản và ngắt runtime có thể khiến lượt chưa hoàn tất. ROUND_COMPLETE chỉ
là hết việc của lượt này. Nhập Outputs về Drive để xuất lượt còn thiếu.
Chỉ checkpoint của version đã lưu thành công mới chuyển qua runtime khác được.
Không dùng dataset hay Outputs của luồng Qwen 700k cũ ở notebook này.

In [ ]:
# Kaggle only. This cell prints before any install or download.
import hashlib
import importlib
import json
import os
from pathlib import Path
import re
import subprocess
import sys
import time
SESSION_STARTED = time.monotonic()
KAGGLE_ACCOUNT_ID = 0
SESSION_HOURS = 10.0
print('NEW DATA Kaggle bootstrap | account', KAGGLE_ACCOUNT_ID, '| checking T4 x2', flush=True)
if Path('/var/colab/hostname').is_file() or not Path('/kaggle/input').is_dir():
    raise RuntimeError('Notebook GPU này chạy trên Kaggle. Colab CPU dùng 04_colab_export_new_data_kaggle.')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=True)
jobs = {}
for path in Path('/kaggle/input').rglob('job.json'):
    value = json.loads(path.read_text())
    if value.get('schema') == 'new-data-kaggle-v1':
        jobs[value.get('manifest_sha256')] = path.parent
if len(jobs) != 1:
    raise RuntimeError('Add Input đúng MỘT dataset NEW_DATASET_READY từ notebook xuất data mới. Gỡ gói 700k/continuation cũ.')
JOB_DIR = next(iter(jobs.values()))
JOB = json.loads((JOB_DIR/'job.json').read_text())
def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(',', ':')).encode()).hexdigest()
if JOB.get('manifest_sha256') != digest({k:v for k,v in JOB.items() if k != 'manifest_sha256'}):
    raise ValueError('Job header changed.')
if JOB.get('repo_url') != 'https://github.com/Platypus27-coder/VietMedBridge.git' or not re.fullmatch('[0-9a-f]{40}', JOB.get('code_commit', '')):
    raise ValueError('Job producer repo/commit invalid.')
if not 0 <= KAGGLE_ACCOUNT_ID < JOB['team']['accounts'] or JOB['team']['gpu_count'] != 2:
    raise ValueError('Account ID không thuộc team đã xuất; không tự đổi team trong notebook GPU.')
expected_sources = {'new_data_kaggle.py','new_data_embeddings.py','new_data_cache_lookup.py','new_data_drive_api.py'}
if set(JOB['worker_sources']) != expected_sources:
    raise ValueError('Missing worker sources.')
for name, checksum in JOB['worker_sources'].items():
    if hashlib.sha256((JOB_DIR/name).read_bytes()).hexdigest() != checksum:
        raise ValueError('Worker source checksum changed.')
CHECKOUT = Path('/kaggle/temp/VietMedBridge-new-data')
WORK_DIR = Path('/kaggle/temp/vmb_new_data')
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(WORK_DIR/'hf_cache')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
if not (CHECKOUT/'.git').is_dir():
    subprocess.run(['git','clone','--no-checkout',JOB['repo_url'],str(CHECKOUT)], check=True)
if subprocess.check_output(['git','-C',str(CHECKOUT),'remote','get-url','origin'], text=True).strip() != JOB['repo_url']:
    raise ValueError('Checkout belongs to another repository.')
subprocess.run(['git','-C',str(CHECKOUT),'fetch','--depth','1','origin',JOB['code_commit']], check=True)
subprocess.run(['git','-C',str(CHECKOUT),'checkout','--detach','FETCH_HEAD'], check=True)
print('Installing pinned encoder dependencies; no Google Drive mount on Kaggle.', flush=True)
subprocess.run([sys.executable,'-m','pip','install','-e','.[strong]'], cwd=CHECKOUT, check=True)
sys.path[:0] = [str(JOB_DIR),str(CHECKOUT/'src')]
importlib.invalidate_caches()
from vietmedbridge.portable_embeddings import runtime_install_commands
for command in runtime_install_commands(JOB['runtime']):
    subprocess.run(command, check=True)
env = os.environ.copy()
env['PYTHONPATH'] = str(CHECKOUT/'src')+os.pathsep+str(JOB_DIR)
probe = "import torch; print('FRESH CUDA:',torch.__version__,torch.cuda.is_available(),torch.cuda.device_count(),flush=True); assert torch.cuda.is_available() and torch.cuda.device_count()==2"
subprocess.run([sys.executable,'-u','-c',probe], env=env, check=True)
print('READY | batch',JOB['batch'],'| team',JOB['team'],'| account',KAGGLE_ACCOUNT_ID,flush=True)

In [ ]:
# Each T4 runs an independent fresh Python worker; output stays in /kaggle/working.
elapsed = (time.monotonic()-SESSION_STARTED)/3600
if elapsed >= SESSION_HOURS-0.1:
    raise RuntimeError('Bootstrap used the save deadline; restart a fresh saved version before encoding.')
command = [sys.executable,'-u',str(JOB_DIR/'new_data_kaggle.py'),'launch',
    '--job',str(JOB_DIR),'--checkout',str(CHECKOUT),
    '--output','/kaggle/working/vmb_new_data','--work',str(WORK_DIR/'workers'),
    '--account-id',str(KAGGLE_ACCOUNT_ID),'--hours',str(SESSION_HOURS-elapsed)]
subprocess.run(command, env=env, check=True)
print('Save & Run All: chờ version hoàn tất rồi nhập Outputs về Drive bằng notebook Colab CPU.',flush=True)